# 02. Config Loading — Layered TOML, Validated, Cached

ArcLLM is config-driven. Every model, every module toggle, every default lives in a TOML file. Code reads typed Pydantic models. Misconfiguration crashes at startup, never mid-flight.

**What v0.4 added:** the global config is now *layered*. Packaged defaults at `<arcllm>/config.toml` are overlaid by user overrides at `${ARC_CONFIG_DIR:-~/.arc}/arcllm.toml`. Dicts deep-merge. Lists and scalars replace. Missing user file is a no-op. This is how a federal operator drops a single TOML on the box and changes behavior across every agent without touching the package.

**You will learn:**
- The shape of a provider TOML (`anthropic.toml`) and what each   section maps to.
- `load_provider_config()` — what it parses, what it validates.
- `load_global_config()` — defaults, modules, vault, layered   precedence.
- The user-overlay mechanism: `ARC_CONFIG_DIR` + `arcllm.toml`.
- Deep-merge semantics: dicts merge, scalars and lists replace.
- The registry-level cache and `clear_cache()` semantics.
- The error surface — missing keys, malformed TOML, invalid   provider names.


## 1. Setup

The first code cell is the standard walkthrough boilerplate: locate the repo root, register every package's `src/` on `sys.path`, and load `.env`. Every notebook in this collection starts identically — paste it once, never think about it again.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Now pull in the public surface this notebook exercises. Everything is exported from the top-level `arcllm` package. The Pydantic models are pure data — instantiating them does not hit the filesystem or the network.

In [ ]:
from arcllm import (
    ArcLLMConfigError,
    DefaultsConfig,
    GlobalConfig,
    ModelMetadata,
    ModuleConfig,
    ProviderConfig,
    ProviderSettings,
    clear_cache,
    load_global_config,
    load_provider_config,
)

# Helpers used throughout the notebook
import shutil
import tempfile
import tomllib
from pathlib import Path

Two utilities make every overlay demo below safe and isolated:

- `set_arc_config_dir(path)` — point the loader at a sandbox   directory by setting `ARC_CONFIG_DIR`.
- A `with_user_overlay(...)` context manager that writes a   user `arcllm.toml`, loads, and tears down on exit.

Critically, every demo calls `clear_cache()` before *and* after to avoid bleeding cached config between cells.

In [ ]:
import contextlib


@contextlib.contextmanager
def with_user_overlay(toml_text: str, env_var: str = "ARC_CONFIG_DIR"):
    """Write a user arcllm.toml into a tmp dir and point ARC_CONFIG_DIR at it."""
    prev = os.environ.get(env_var)
    tmp = tempfile.mkdtemp(prefix="arc-overlay-")
    try:
        os.environ[env_var] = tmp
        (Path(tmp) / "arcllm.toml").write_text(toml_text)
        clear_cache()
        yield Path(tmp)
    finally:
        if prev is None:
            os.environ.pop(env_var, None)
        else:
            os.environ[env_var] = prev
        shutil.rmtree(tmp, ignore_errors=True)
        clear_cache()

## 2. Provider TOML structure

Each provider lives in its own TOML file under `packages/arcllm/src/arcllm/providers/`. There are sixteen of them as of v0.4: anthropic, openai, azure_openai, google, cohere, xai, deepseek, groq, mistral, moonshot, fireworks, together, ollama, vllm, huggingface, huggingface_tgi.

A provider TOML has exactly two top-level sections:

- `[provider]` — connection settings (api format, base URL, env   var holding the key, default model, default temperature).
- `[models.<name>]` — one block per model, each with capability   flags (tools, vision, thinking) and pricing per million   tokens (input, output, cache read, cache write).

API keys themselves are *never* in the TOML — only the env var name is. That's how the security layer keeps secrets off disk.

In [ ]:
# Locate the packaged providers/ directory and list what's there
import arcllm

PROVIDERS_DIR = Path(arcllm.__file__).parent / "providers"
provider_files = sorted(p.stem for p in PROVIDERS_DIR.glob("*.toml"))
print(f"Packaged providers: {len(provider_files)}")
for name in provider_files:
    print(f"  {name}")

In [ ]:
# Read the Anthropic TOML raw — bypass the loader so we can see
# what the file actually contains before validation.
anthropic_path = PROVIDERS_DIR / "anthropic.toml"
with open(anthropic_path, "rb") as f:
    raw = tomllib.load(f)

print("Top-level keys:", list(raw.keys()))
print("provider keys:", list(raw["provider"].keys()))
print("model count:", len(raw["models"]))
print("first model:", next(iter(raw["models"])))

Note that the model dict is keyed by *the actual API model id* — `claude-sonnet-4-6`, not a friendly alias. This is a deliberate choice: the TOML is the source of truth for what the wire call sends. There's no remapping layer to confuse a compliance auditor reading logs.

## 3. `load_provider_config` — packaged defaults

`load_provider_config(name)` does four things, in order:

1. Validates `name` against `^[a-z][a-z0-9_]*$` (max 64 chars).    This blocks path traversal and arbitrary module injection —    ASI-04 / NIST SI-10.
2. Resolves the path: `<arcllm>/providers/<name>.toml`.
3. Parses with `tomllib` (stdlib, zero deps).
4. Validates the dict against `ProviderConfig` and returns a    typed model.

Any failure in steps 2–4 raises `ArcLLMConfigError`. There is no partial result. There is no fallback. Fail closed.

In [ ]:
anthropic = load_provider_config("anthropic")

print(type(anthropic).__name__)
print("default_model:    ", anthropic.provider.default_model)
print("default_temp:     ", anthropic.provider.default_temperature)
print("api_key_env:      ", anthropic.provider.api_key_env)
print("api_key_required: ", anthropic.provider.api_key_required)
print("base_url:         ", anthropic.provider.base_url)
print("api_format:       ", anthropic.provider.api_format)
print("models:           ", len(anthropic.models))

In [ ]:
# ProviderSettings has a field-level validator that enforces HTTPS
# for non-localhost hosts. Try to construct one with a plain http://
# remote URL and watch it reject:
from pydantic import ValidationError

try:
    ProviderSettings(
        api_format="anthropic-messages",
        base_url="http://api.anthropic.com",  # <-- not localhost
        api_key_env="ANTHROPIC_API_KEY",
        default_model="claude-sonnet-4-6",
        default_temperature=0.7,
    )
except ValidationError as e:
    print("ValidationError:", str(e).splitlines()[-1])

# But localhost is fine — handy for vLLM, Ollama, dev backends:
ok = ProviderSettings(
    api_format="openai-chat",
    base_url="http://localhost:11434",
    api_key_env="",
    api_key_required=False,
    default_model="llama3",
    default_temperature=0.7,
)
print("localhost ok:", ok.base_url)

Each model's metadata is a `ModelMetadata` instance. Every field is required — there are no Optional fields, no `None` sentinels, no "figure it out at runtime" gaps. If a model doesn't support tools, the TOML must say so explicitly.

In [ ]:
# Inspect a single model
sonnet = anthropic.models["claude-sonnet-4-6"]
print(type(sonnet).__name__)
print("context_window:        ", sonnet.context_window)
print("max_output_tokens:     ", sonnet.max_output_tokens)
print("supports_tools:        ", sonnet.supports_tools)
print("supports_vision:       ", sonnet.supports_vision)
print("supports_thinking:     ", sonnet.supports_thinking)
print("input_modalities:      ", sonnet.input_modalities)
print("cost_input_per_1m:     ", sonnet.cost_input_per_1m)
print("cost_output_per_1m:    ", sonnet.cost_output_per_1m)
print("cost_cache_read_per_1m:", sonnet.cost_cache_read_per_1m)

In [ ]:
# All models, sorted by input cost, with a tools/vision/thinking flag column
rows = sorted(
    anthropic.models.items(),
    key=lambda kv: kv[1].cost_input_per_1m,
)
print(f"{'model':35s} {'in/M':>6s} {'out/M':>6s}  T V Th")
print("-" * 65)
for name, meta in rows:
    print(
        f"{name:35s} "
        f"{meta.cost_input_per_1m:6.2f} "
        f"{meta.cost_output_per_1m:6.2f}  "
        f"{int(meta.supports_tools)} "
        f"{int(meta.supports_vision)} "
        f"{int(meta.supports_thinking)}"
    )

Two things to notice. First, the model id is the dictionary key — so a routing module can do `cfg.models[chosen]` directly with no translation. Second, `supports_thinking=True` is the flag the OpenAI adapter keys on to remap `system` → `developer` for o-series models. A capability bit in TOML drives behavior in the adapter. That's the clean-separation pattern at work.

## 4. User overlay precedence — `ARC_CONFIG_DIR` + `arcllm.toml`

Layered config is the v0.4 headline feature. The contract:

1. Loader reads packaged `<arcllm>/config.toml` as the base.
2. If `${ARC_CONFIG_DIR:-~/.arc}/arcllm.toml` exists, parse it.
3. Deep-merge user data on top of base data.
4. Validate the merged dict and return a typed `GlobalConfig`.

Missing user file is a *silent no-op* — no warning, no error. That's intentional: the packaged defaults must always work, even on a fresh box where nobody dropped a user TOML.

**Layering applies only to the global config.** Provider TOMLs are loaded directly from the package and are not user-overridable through this mechanism. The reason: provider model metadata is the source of truth for compliance — pricing, capabilities, context windows. An operator should not be able to silently change those out from under an audit log.

In [ ]:
# Baseline: what's in the packaged config?
clear_cache()
base = load_global_config()
print("defaults.provider:    ", base.defaults.provider)
print("defaults.temperature: ", base.defaults.temperature)
print("defaults.max_tokens:  ", base.defaults.max_tokens)
print("modules:              ", sorted(base.modules.keys()))
print("modules.telemetry.enabled:", base.modules["telemetry"].enabled)
print("modules.audit.enabled:    ", base.modules["audit"].enabled)

Now the overlay demo. Write a user TOML that flips a couple of scalars, point `ARC_CONFIG_DIR` at the temp dir, reload, and verify the merged result.

In [ ]:
user_toml = """
[defaults]
temperature = 0.0

[modules.audit]
enabled = true
"""

with with_user_overlay(user_toml):
    overlaid = load_global_config()

print(
    "temperature  base -> overlay:", base.defaults.temperature, "->", overlaid.defaults.temperature
)
print(
    "audit.enabled base -> overlay:",
    base.modules["audit"].enabled,
    "->",
    overlaid.modules["audit"].enabled,
)
# Untouched fields fall through unchanged.
print("max_tokens (untouched):       ", overlaid.defaults.max_tokens)
print("provider (untouched):         ", overlaid.defaults.provider)
print("telemetry.enabled (untouched):", overlaid.modules["telemetry"].enabled)

The path resolution honors `~` (home expansion) too. If `ARC_CONFIG_DIR` is unset, the loader looks at `~/.arc/arcllm.toml`. When it *is* set, the value is expanduser-ed before lookup, so `ARC_CONFIG_DIR=~/etc/arc` resolves correctly across user accounts.

In [ ]:
# Verify the path resolution rule from source:
from arcllm.config import _user_config_path

# With ARC_CONFIG_DIR unset, loader looks at ~/.arc/arcllm.toml
os.environ.pop("ARC_CONFIG_DIR", None)
expected_default = Path.home() / ".arc" / "arcllm.toml"
actual_default = _user_config_path()
# Returns None if file doesn't exist; otherwise returns the path
print("default user path expected at:", expected_default)
print("loader returned (None if absent):", actual_default)

# With ARC_CONFIG_DIR set explicitly, the loader uses it.
with tempfile.TemporaryDirectory() as tmp:
    os.environ["ARC_CONFIG_DIR"] = tmp
    (Path(tmp) / "arcllm.toml").write_text("[defaults]\n")
    resolved = _user_config_path()
    print("explicit ARC_CONFIG_DIR resolved:", resolved)
    os.environ.pop("ARC_CONFIG_DIR", None)

A user file with no overlapping keys is harmless — it just adds nothing. Below, the user TOML touches a single nested module key and leaves everything else inherited.

In [ ]:
# Surgical override: change only retry.max_retries.
minimal = """
[modules.retry]
max_retries = 99
"""

with with_user_overlay(minimal):
    cfg = load_global_config()

retry = cfg.modules["retry"]
print("retry.enabled (inherited):     ", retry.enabled)
print("retry.max_retries (overridden):", retry.max_retries)
print("retry.backoff_base_seconds (inherited):", retry.backoff_base_seconds)

## 5. Deep-merge semantics — dicts merge, scalars and lists replace

The merge rule is short and worth memorizing:

- **Both sides are dicts** → recurse, merging key-by-key.
- **Anything else** → user value wins, base value is discarded.

That second rule is important for *lists*. The fallback chain, for example, is a list. If your user TOML sets `fallback.chain = ["groq"]`, you get exactly `["groq"]` — not the union of `["anthropic", "openai"]` and `["groq"]`. List union is almost never what you want for an ordered policy like a fallback chain.

In [ ]:
from arcllm.config import _deep_merge

# Dict merge: nested keys preserved
a = {"defaults": {"provider": "anthropic", "temperature": 0.7}}
b = {"defaults": {"temperature": 0.0}}
print("dict merge:", _deep_merge(a, b))

In [ ]:
# List replace: user list wins outright
a = {"chain": ["anthropic", "openai"]}
b = {"chain": ["groq"]}
print("list replace:", _deep_merge(a, b))

In [ ]:
# Scalar replace
a = {"max_retries": 3}
b = {"max_retries": 99}
print("scalar replace:", _deep_merge(a, b))

# Type change is allowed — user wins regardless of base type.
a = {"timeout": 30}
b = {"timeout": "30s"}
print("type-change replace:", _deep_merge(a, b))

In [ ]:
# Nested dict + nested scalar override
base_data = {
    "defaults": {"provider": "anthropic", "temperature": 0.7, "max_tokens": 4096},
    "modules": {
        "telemetry": {"enabled": True, "log_level": "INFO"},
        "fallback": {"enabled": False, "chain": ["anthropic", "openai"]},
    },
}
user_data = {
    "defaults": {"temperature": 0.0},
    "modules": {
        "telemetry": {"log_level": "DEBUG"},
        "fallback": {"enabled": True, "chain": ["groq"]},
    },
}

merged = _deep_merge(base_data, user_data)
print("defaults:        ", merged["defaults"])
print("telemetry:       ", merged["modules"]["telemetry"])
print("fallback:        ", merged["modules"]["fallback"])

Walk through what happened above:

- `defaults.provider` survived because the user dict didn't   mention it.
- `defaults.temperature` was replaced by the user's 0.0.
- `defaults.max_tokens` was preserved (untouched).
- `modules.telemetry.enabled` survived; `log_level` was   replaced from `INFO` → `DEBUG`.
- `modules.fallback.chain` was *fully replaced* — `["groq"]`,   not `["anthropic", "openai", "groq"]`.

If list-union semantics are what you actually need, do it explicitly in code outside the loader. The loader is intentionally dumb and predictable.

## 6. `load_global_config` — `GlobalConfig`, `DefaultsConfig`, `ModuleConfig`

`GlobalConfig` has three fields:

- `defaults: DefaultsConfig` — provider name, temperature,   max_tokens. All have package-level defaults; you don't have   to specify them.
- `modules: dict[str, ModuleConfig]` — one entry per module.   `ModuleConfig` is `extra='allow'`, so each module's TOML   section can carry its own bespoke fields without changing   the schema.
- `vault: VaultConfig` — optional vault backend settings.

The `extra='allow'` on `ModuleConfig` is load-bearing. It's the reason a new module can ship with new config fields and not require a coordinated release of `arcllm.config`.

In [ ]:
clear_cache()
g = load_global_config()

print(type(g).__name__)
print("defaults type:", type(g.defaults).__name__)
print("modules keys: ", sorted(g.modules.keys()))
print("vault type:   ", type(g.vault).__name__)

In [ ]:
# DefaultsConfig — every field has a sensible default,
# so you can construct one with no kwargs.
empty = DefaultsConfig()
print("empty DefaultsConfig:")
print("  provider:   ", empty.provider)
print("  temperature:", empty.temperature)
print("  max_tokens: ", empty.max_tokens)

In [ ]:
# ModuleConfig with extra fields preserved via extra="allow"
basic = ModuleConfig(enabled=True)
extra = ModuleConfig(
    enabled=True,
    max_concurrent=8,
    call_timeout=120.0,
    custom_label="redacted-cui",
)

print("basic dump:", basic.model_dump())
print("extra dump:", extra.model_dump())
print("attribute access on extras:", extra.max_concurrent, extra.custom_label)

In [ ]:
# Real-world: telemetry config from the packaged file carries an
# extra log_level field that DefaultsConfig wouldn't accept.
tele = g.modules["telemetry"]
print("telemetry enabled:  ", tele.enabled)
print("telemetry log_level:", tele.log_level)
print("queue extras:       ", g.modules["queue"].model_dump())

Notice `queue.call_timeout = 180.0` in the dump. That's the v0.4 default — extended from 60s because real federal deployments saw legitimate multi-tool turns get cancelled mid-flight on the old budget. The TOML carries the why in a comment; the loader carries it forward via `extra='allow'`.

End-to-end demo: drop a user TOML, observe defaults flow through, observe overrides land, observe a brand-new module config (e.g., a hypothetical `cuda` module) round-trip cleanly.

In [ ]:
future_module = """
[modules.cuda]
enabled = true
visible_devices = "0,1"
memory_fraction = 0.85
"""

with with_user_overlay(future_module):
    cfg = load_global_config()
    cuda = cfg.modules["cuda"]
    print("cuda.enabled:        ", cuda.enabled)
    print("cuda.visible_devices:", cuda.visible_devices)
    print("cuda.memory_fraction:", cuda.memory_fraction)
    print("type:                ", type(cuda).__name__)

## 7. Cache and `clear_cache()` invalidation

The provider config and global config are cached at the *registry* layer (one level above the loaders). The functions in `arcllm.config` are pure — they always re-read disk. The registry holds a memoization layer so `load_model()` doesn't re-parse TOML on every call.

`clear_cache()` (exported from the top-level `arcllm` package) wipes:

- The provider config cache (`_provider_config_cache`).
- The adapter class cache (`_adapter_class_cache`).
- The cached `GlobalConfig`.
- Per-module state: rate-limit buckets, OTel SDK, telemetry   budgets, telemetry global defaults.

Tests call it between cases. Notebooks should call it before and after any overlay demo. Production code generally does not need to.

In [ ]:
from arcllm import registry

clear_cache()
# First call populates the cache
first = load_provider_config("anthropic")
print(
    "after first call - cached providers:",
    list(registry._provider_config_cache.keys())
    if registry._provider_config_cache
    else "(loader does not populate registry cache)",
)

# load_model() is what drives the registry cache; loader calls
# don't necessarily fill it. But clear_cache() wipes either way.
clear_cache()
print("after clear_cache - cached providers:", list(registry._provider_config_cache.keys()))

In [ ]:
# Demonstrate cache invalidation around an overlay change.
# Without clear_cache(), a stale GlobalConfig could survive.
clear_cache()
before = load_global_config()

with with_user_overlay("[defaults]\ntemperature = 0.123\n"):
    # with_user_overlay calls clear_cache() on entry, so the
    # next load_global_config() picks up the overlay.
    inside = load_global_config()

# After the contextmanager exits, ARC_CONFIG_DIR is restored
# AND clear_cache() is called again — so the next load returns
# the packaged defaults, not the overlay.
after = load_global_config()

print("temperature timeline:")
print("  before overlay:", before.defaults.temperature)
print("  inside overlay:", inside.defaults.temperature)
print("  after  overlay:", after.defaults.temperature)

Why both module-level caches *and* a manual reset? Because module state isn't config — it's *process state* (open buckets, OTel exporters, accumulated budgets). Re-reading the TOML doesn't reset those. `clear_cache()` does.

## 8. Common errors — missing key, malformed TOML, invalid name

Every error path in the config loader raises `ArcLLMConfigError`, which inherits from `ArcLLMError`. So callers can catch at whatever granularity they want — narrow (just config errors) or wide (any arcllm error).

In [ ]:
# 1. Missing provider file
try:
    load_provider_config("totally_made_up_provider")
except ArcLLMConfigError as e:
    print("missing file ->", str(e).split(":")[0])

In [ ]:
# 2. Path-traversal attempt — blocked by name regex.
for bad_name in [
    "../etc/passwd",
    "..",
    "anthropic/../../../etc",
    "Anthropic",  # uppercase rejected (regex is lowercase-only)
    "1openai",  # must start with a letter
    "open ai",  # spaces not allowed
    "open-ai",  # hyphen not allowed (only [a-z0-9_])
    "",  # empty
    "a" * 65,  # too long (>64)
]:
    try:
        load_provider_config(bad_name)
    except ArcLLMConfigError as e:
        snippet = bad_name if len(bad_name) <= 30 else bad_name[:27] + "..."
        first_line = str(e).splitlines()[0]
        print(f"{snippet!r:35s} -> {first_line[:70]}")

In [ ]:
# 3. Malformed TOML in a user overlay.
broken = "[defaults\nthis is not valid toml\n"

try:
    with with_user_overlay(broken):
        load_global_config()
except ArcLLMConfigError as e:
    print("malformed ->", str(e).splitlines()[0][:90])

In [ ]:
# 4. Wrong types in user overlay get caught by Pydantic.
wrong_types = """
[defaults]
temperature = "not-a-float"
"""

try:
    with with_user_overlay(wrong_types):
        load_global_config()
except ArcLLMConfigError as e:
    msg = str(e)
    print("validation -> first 120 chars:")
    print(" ", msg[:120].replace("\n", " "))

In [ ]:
# 5. Provider file with missing required model field.
# Build a temp providers/<x>.toml that's missing context_window.
import arcllm

broken_provider_path = Path(arcllm.__file__).parent / "providers" / "_broken_demo.toml"
broken_provider_path.write_text(
    "[provider]\n"
    'api_format = "openai-chat"\n'
    'base_url = "https://example.invalid"\n'
    'api_key_env = "X_API_KEY"\n'
    'default_model = "x"\n'
    "default_temperature = 0.7\n"
    "\n"
    "[models.x]\n"
    "context_window = 1000\n"  # everything else missing
)
try:
    clear_cache()
    load_provider_config("_broken_demo")
except ArcLLMConfigError as e:
    print("missing required fields -> first 200 chars:")
    print(" ", str(e)[:200].replace("\n", " "))
finally:
    broken_provider_path.unlink(missing_ok=True)
    clear_cache()

Every one of those failures happens at *load* time, before any LLM call goes out. That's the entire reason this layer exists. An autonomous agent that crashes at startup is a bug; an agent that crashes three hours into a batch run because a pricing field was a string is a fire.

## 9. Summary

**What this notebook covered:**

- Provider TOML structure: `[provider]` + `[models.*]`. Sixteen   packaged providers as of v0.4.
- `load_provider_config(name)`: name validated against   `^[a-z][a-z0-9_]*$`, file resolved package-relative, parsed,   validated as `ProviderConfig`.
- `load_global_config()`: layered. Packaged   `<arcllm>/config.toml` is the base;   `${ARC_CONFIG_DIR:-~/.arc}/arcllm.toml` deep-merges over it.
- Deep-merge rule: dicts recurse, scalars and lists replace.   No list-union magic.
- `GlobalConfig` (`DefaultsConfig` + `dict[str, ModuleConfig]`   + `VaultConfig`); `ModuleConfig` is `extra='allow'` so new   modules can ship new fields without coordinated releases.
- Cache invalidation: `arcllm.clear_cache()` wipes provider   config, adapter class, global config, and per-module state   (rate-limit buckets, OTel SDK, telemetry budgets).
- Failure modes: missing files, path-traversal names,   malformed TOML, validation errors — all surface as   `ArcLLMConfigError` at load time.

**Public API exercised:**

- `load_global_config`, `load_provider_config`
- `GlobalConfig`, `ProviderConfig`
- `DefaultsConfig`, `ModuleConfig`
- `ProviderSettings`, `ModelMetadata`
- `clear_cache`
- `ArcLLMConfigError`

**Up next:** notebook `03-anthropic-adapter.ipynb` — how the Anthropic adapter consumes a `ProviderConfig` to construct and parse Messages-API requests.